# Online Retail II Exploration

This notebook is the **exploration layer** of the pipeline: it loads both yearly sheets of the
source workbook, works through every data-quality issue in the raw export, and documents the
finding and decision behind each one (duplicate sheet overlap, cancellations, stock write-offs,
free items, bad debt, duplicate lines, non-product codes, multiple descriptions per product,
non-country values, the partial final month).

It never writes to `data/` and never touches production data directly — its output is
`reports/analysis_report.md` (hand-written from this notebook's figures, re-generated by
actually re-running this notebook end to end whenever the source data or a decision changes) and,
more importantly, the *decisions* themselves. Every decision this notebook reaches becomes a
concrete step in `power_query_cleaning_guide.md`, which is the actual production spec: that guide
is what gets built into the Power Query / Power BI model in `reports/`, so the live report can
refresh itself as new monthly files land. Think of it as research → spec → build: this notebook
is the research.


## 1. Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Configure File Paths

In [2]:
PROJECT_ROOT = Path.cwd()
INPUT_PATH = PROJECT_ROOT / "data" / "online_retail_II.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "reports"

INPUT_PATH, OUTPUT_DIR

(WindowsPath('C:/Users/sezen/Projects/RetailDemo/retail-self-refreshing-report/data/online_retail_II.xlsx'),
 WindowsPath('C:/Users/sezen/Projects/RetailDemo/retail-self-refreshing-report/reports'))

## 3. Read Excel Workbook into a DataFrame

In [3]:
sheets = pd.read_excel(INPUT_PATH, sheet_name=None)
df = pd.concat(
    [sheet.assign(source_sheet=sheet_name) for sheet_name, sheet in sheets.items()],
    ignore_index=True,
)

print(f"Loaded {len(sheets)} sheets and {len(df):,} rows.")
df.head()

Loaded 2 sheets and 1,067,371 rows.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


## 3b. Remove Duplicate Rows from Overlapping Sheet Date Ranges

In [4]:
# The "Year 2009-2010" sheet runs through 2010-12-09, overlapping the first 9 days of the
# "Year 2010-2011" sheet (which starts 2010-12-01). Invoices in that window are recorded
# identically in both sheets, so concatenating them as-is double-counts that period's revenue.
overlap_invoices = (
    df.groupby("Invoice")["source_sheet"].nunique().loc[lambda s: s > 1].index
)
overlap_row_mask = df["Invoice"].isin(overlap_invoices) & (df["source_sheet"] == "Year 2009-2010")

print(f"Invoices present in both sheets: {len(overlap_invoices):,}")
print(f"Rows dropped (the 'Year 2009-2010' copy of those invoices): {overlap_row_mask.sum():,}")

df = df.loc[~overlap_row_mask].reset_index(drop=True)
print(f"Rows remaining: {len(df):,}")

Invoices present in both sheets: 1,088
Rows dropped (the 'Year 2009-2010' copy of those invoices): 22,523
Rows remaining: 1,044,848


## 4. Inspect the DataFrame

In [5]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.info()
df.head(10)

Shape: (1044848, 9)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'source_sheet']


<class 'pandas.DataFrame'>
RangeIndex: 1044848 entries, 0 to 1044847
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   Invoice       1044848 non-null  object        
 1   StockCode     1044848 non-null  object        
 2   Description   1040573 non-null  object        
 3   Quantity      1044848 non-null  int64         
 4   InvoiceDate   1044848 non-null  datetime64[us]
 5   Price         1044848 non-null  float64       
 6   Customer ID   809561 non-null   float64       
 7   Country       1044848 non-null  str           
 8   source_sheet  1044848 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(2)
memory usage: 71.7+ MB


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,Year 2009-2010
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,Year 2009-2010
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,Year 2009-2010
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom,Year 2009-2010


## 5. Validate Expected Columns

In [6]:
expected_columns = {
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
    "source_sheet",
}
missing_columns = expected_columns.difference(df.columns)
extra_columns = set(df.columns).difference(expected_columns)

assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"
print("Schema is valid.")
print("Extra columns:", sorted(extra_columns) if extra_columns else "None")

Schema is valid.
Extra columns: None


## 6. Convert Data Types

In [7]:
# Invoice and StockCode mix int and str values in the source file (purely-numeric values get
# read as ints, e.g. 492525 or 85123, while anything needing a letter -- "C489449", "85123A",
# "AMAZONFEE" -- comes in as str). Casting both to string once, here, means every downstream
# comparison/regex/prefix-check on these columns behaves consistently for the rest of the notebook.
df["Invoice"] = df["Invoice"].astype("string")
df["StockCode"] = df["StockCode"].astype("string")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Revenue"] = df["Quantity"] * df["Price"]

df[["Invoice", "StockCode", "Quantity", "Price", "InvoiceDate", "Revenue"]].dtypes

Invoice                string
StockCode              string
Quantity                int64
Price                 float64
InvoiceDate    datetime64[us]
Revenue               float64
dtype: object

## 6b. Remove Literal Test Rows (`TEST001` / `TEST002`)

In [8]:
# TEST001/TEST002 are literal QA/test rows (Description "This is a test product."), not real
# transactions -- they show up scattered across normal_sale, free_item, cancelled, and
# stock_writein purely because of whatever sign their individual rows happen to carry, not
# because they represent genuine sales activity (see §7.8 in the report). Dropping them here,
# before any of the quality/category analysis below, keeps every downstream count and total
# based on real transactions only.
test_code_mask = df["StockCode"].isin(["TEST001", "TEST002"])
print(f"Test-product rows removed: {test_code_mask.sum():,}")

df = df.loc[~test_code_mask].reset_index(drop=True)
print(f"Rows remaining: {len(df):,}")

Test-product rows removed: 17
Rows remaining: 1,044,831


In [9]:
def column_extremes(column):
    values = df[column].dropna()
    if values.empty:
        return None, None
    if pd.api.types.is_object_dtype(values) or pd.api.types.is_string_dtype(values):
        values = values.astype("string")
    return values.min(), values.max()

extremes = [column_extremes(column) for column in df.columns]
extreme_values = pd.DataFrame(
    {
        "column": df.columns,
        "minimum": [minimum for minimum, _ in extremes],
        "maximum": [maximum for _, maximum in extremes],
        "non_null_values": [df[column].notna().sum() for column in df.columns],
    }
)

extreme_values

,column,minimum,maximum,non_null_values
0,Invoice,489434,C581569,1044831
1,StockCode,10002,m,1044831
2,Description,DOORMAT UNION JACK GUNS AND ROSES,wrongly sold sets,1040557
3,Quantity,-80995,80995,1044831
4,InvoiceDate,2009-12-01 07:45:00,2011-12-09 12:50:00,1044831
5,Price,-53594.36,38970.0,1044831
6,Customer ID,12346.0,18287.0,809545
7,Country,Australia,West Indies,1044831
8,source_sheet,Year 2009-2010,Year 2010-2011,1044831
9,Revenue,-168469.6,168469.6,1044831


In [10]:
# The Quantity/Revenue extremes above are exactly +-80,995 -- check whether that's one order
# placed and then fully cancelled (as claimed in the report), or two unrelated coincidences.
extreme_rows = df.loc[df["Quantity"].abs() == 80995]
print(extreme_rows[["Invoice", "StockCode", "Description", "Quantity", "Price", "Revenue", "InvoiceDate"]].to_string(index=False))
print()
print("Combined net revenue from these two rows:", extreme_rows["Revenue"].sum())

Invoice StockCode                 Description  Quantity  Price   Revenue         InvoiceDate
 581483     23843 PAPER CRAFT , LITTLE BIRDIE     80995   2.08  168469.6 2011-12-09 09:15:00
C581484     23843 PAPER CRAFT , LITTLE BIRDIE    -80995   2.08 -168469.6 2011-12-09 09:27:00

Combined net revenue from these two rows: 0.0


## 7. Analyze Meaningless Numeric Values

In [11]:
meaningless_masks = {
    "negative_quantity": df["Quantity"] < 0,
    "zero_quantity": df["Quantity"] == 0,
    "negative_price": df["Price"] < 0,
    "zero_price": df["Price"] == 0,
    "negative_revenue": df["Revenue"] < 0,
    "zero_revenue": df["Revenue"] == 0,
}

meaningless_value_summary = pd.DataFrame(
    [
        {
            "issue": issue,
            "column": issue.rsplit("_", 1)[1].title(),
            "count": mask.sum(),
            "percentage_of_non_null": round(mask.sum() / df[column].notna().sum() * 100, 2),
        }
        for issue, mask in meaningless_masks.items()
        for column in [issue.rsplit("_", 1)[1].title()]
    ]
)

meaningless_flag_mask = pd.DataFrame(meaningless_masks).any(axis=1)
meaningless_rows = df.loc[meaningless_flag_mask].copy()
meaningless_rows["meaningless_flags"] = (
    pd.DataFrame(meaningless_masks, index=df.index)
    .loc[meaningless_flag_mask]
    .apply(lambda row: ", ".join(row.index[row]), axis=1)
)

display(meaningless_value_summary)
display(meaningless_rows.head(20))

,issue,column,count,percentage_of_non_null
0,negative_quantity,Quantity,22553,2.16
1,zero_quantity,Quantity,0,0.00
2,negative_price,Price,5,0.00
3,zero_price,Price,6021,0.58
4,negative_revenue,Revenue,19165,1.83
5,zero_revenue,Revenue,6021,0.58


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,Revenue,meaningless_flags
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010,-35.40,"negative_quantity, negative_revenue"
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,Year 2009-2010,-9.90,"negative_quantity, negative_revenue"
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,Year 2009-2010,-17.00,"negative_quantity, negative_revenue"
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,Year 2009-2010,-12.60,"negative_quantity, negative_revenue"
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010,-35.40,"negative_quantity, negative_revenue"
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia,Year 2009-2010,-15.00,"negative_quantity, negative_revenue"
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia,Year 2009-2010,-15.00,"negative_quantity, negative_revenue"
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia,Year 2009-2010,-20.40,"negative_quantity, negative_revenue"
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010,-35.40,"negative_quantity, negative_revenue"
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom,Year 2009-2010,-12.75,"negative_quantity, negative_revenue"


## 8. Identify Invalid Rows

In [12]:
df["is_cancelled"] = df["Invoice"].str.upper().str.startswith("C", na=False)

missing_invoice = df["Invoice"].isna()
missing_stock_code = df["StockCode"].isna()
missing_price = df["Price"].isna()
non_numeric_quantity = df["Quantity"].isna()
invalid_date = df["InvoiceDate"].isna()

invalid_mask = (
    missing_invoice
    | missing_stock_code
    | missing_price
    | non_numeric_quantity
    | invalid_date
)
invalid_rows = df.loc[invalid_mask].copy()
invalid_rows["quality_flag"] = "invalid_core_data"
invalid_rows.loc[df.loc[invalid_mask, "Customer ID"].isna(), "quality_flag"] = "missing_customer_id"

print(f"Flagged rows: {len(invalid_rows):,}")
invalid_rows.head()

Flagged rows: 0


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,Revenue,is_cancelled,quality_flag


In [13]:
# Do cancellation invoices reference an existing original invoice? If stripping the "C" from a
# cancellation invoice number matched a real prior invoice, cancellations could be linked back
# to the specific order each one reverses.
cancel_invoices = df.loc[df["is_cancelled"], "Invoice"].unique()
stripped = pd.Series(cancel_invoices).str.upper().str[1:]
matches = stripped.isin(set(df["Invoice"])).sum()
print(f"Distinct cancellation invoices: {len(cancel_invoices):,}")
print(f"Stripped-prefix matches found among all Invoice values: {matches}")
print()

# Cancellations should always be negative-quantity by definition (that's the whole point of a
# reversal) -- check for exceptions.
positive_qty_cancellations = df.loc[df["is_cancelled"] & (df["Quantity"] > 0)]
print(f"Cancelled rows with POSITIVE quantity (should be none): {len(positive_qty_cancellations)}")
print(positive_qty_cancellations[["Invoice", "StockCode", "Description", "Quantity", "Price"]].to_string(index=False))

Distinct cancellation invoices: 8,288


Stripped-prefix matches found among all Invoice values: 0

Cancelled rows with POSITIVE quantity (should be none): 1
Invoice StockCode Description  Quantity  Price
C496350         M      Manual         1 373.57


## 8b. Analyze Negative-Quantity Rows That Are Not Cancellations (Stock Adjustments)

In [14]:
negative_noncancel_mask = (df["Quantity"] < 0) & ~df["is_cancelled"]
negative_noncancel = df.loc[negative_noncancel_mask].copy()

print(f"Negative-quantity, non-cancellation rows: {len(negative_noncancel):,}")
print()
print("Price value counts (all rows):")
print(negative_noncancel["Price"].value_counts(dropna=False))
print()
print("Missing Customer ID:", negative_noncancel["Customer ID"].isna().sum(), "of", len(negative_noncancel))
print()
print("Rows per invoice (do these ever share an invoice with a normal sale row?):")
mixed_invoices = set(negative_noncancel["Invoice"]) & set(df.loc[df["Quantity"] > 0, "Invoice"])
print("Invoices also containing a positive-quantity row:", len(mixed_invoices), "of", negative_noncancel["Invoice"].nunique())
print()
print("Top Description values (operational notes, not product names):")
print(negative_noncancel["Description"].value_counts(dropna=False).head(15))

# These rows carry Price == 0 and no Customer ID in every case, are never mixed into a
# normal order invoice, and their free-text Descriptions ("damages", "lost", "missing",
# "thrown away", "check", etc.) read as warehouse stock-control notes rather than
# customer activity. Flag them as stock adjustments, distinct from customer cancellations.
df["is_stock_adjustment"] = negative_noncancel_mask

print()
print("Flagged as is_stock_adjustment:", int(df["is_stock_adjustment"].sum()))
negative_noncancel.head(20)

# A small subset use non-standard StockCodes (not the normal 5-digit-plus-suffix product
# pattern) instead -- flagged here as a follow-up, not yet resolved (see §7.2 "Next step").
is_standard_code = negative_noncancel["StockCode"].str.match(r"^\d{5}[A-Za-z]{0,4}$", na=False)
print()
print(f"Stock-adjustment rows using a non-standard StockCode: {(~is_standard_code).sum()} of {len(negative_noncancel):,}")
print(negative_noncancel.loc[~is_standard_code, "StockCode"].value_counts().to_string())

Negative-quantity, non-cancellation rows: 3,393

Price value counts (all rows):
Price
0.0    3393
Name: count, dtype: int64

Missing Customer ID: 3393 of 3393

Rows per invoice (do these ever share an invoice with a normal sale row?):


Invoices also containing a positive-quantity row:

 0 of 3393

Top Description values (operational notes, not product names):
Description
NaN                       2633
check                      121
damages                     83
?                           81
damaged                     78
missing                     27
sold as set on dotcom       20
Damaged                     17
smashed                      9
thrown away                  9
Unsaleable, destroyed.       9
dotcom                       8
damages?                     7
??                           7
crushed                      6
Name: count, dtype: int64

Flagged as is_stock_adjustment: 3393

Stock-adjustment rows using a non-standard StockCode: 31 of 3,393
StockCode
DCGSSGIRL    1
DCGS0006     1
DCGS0016     1
DCGS0027     1
DCGS0036     1
DCGS0039     1
DCGS0060     1
DCGS0056     1
DCGS0059     1
GIFT         1
DCGSLBOY     1
DCGS0053     1
DCGS0004     1
DCGS0062     1
DCGS0037     1
DCGSSBOY     1
DCGSLGIRL    1
C3           1
SP1002       1
DCGS0055     1
DCGS007

## 8c. Analyze Zero-Price, Positive-Quantity Rows (Stock Write-ins vs. Free Items)

In [15]:
zero_price_pos_qty_mask = (df["Price"] == 0) & (df["Quantity"] > 0) & ~df["is_cancelled"] & ~df["is_stock_adjustment"]
zero_price_pos_qty = df.loc[zero_price_pos_qty_mask].copy()

print(f"Zero-price, positive-quantity, non-cancelled, non-stock-adjustment rows: {len(zero_price_pos_qty):,}")
print()

has_customer = zero_price_pos_qty["Customer ID"].notna()
print("Split by Customer ID presence:")
print(has_customer.value_counts())
print()

print("--- Group 1: no Customer ID (stock write-in candidates) ---")
group1 = zero_price_pos_qty.loc[~has_customer]
print(f"Rows: {len(group1):,}")
print("Description value counts:")
print(group1["Description"].value_counts(dropna=False).head(15))
print()

# Do write-in rows ever land on the same invoice as a normally priced row? If so, excluding the
# write-in line can't be double-counted or lose revenue, since the invoice's real total lives on
# its other, priced rows.
mixed_writein_invoices = set(group1["Invoice"]) & set(df.loc[df["Price"] > 0, "Invoice"])
print(f"Write-in rows sharing an invoice with a normally priced row: {len(mixed_writein_invoices):,} of {group1['Invoice'].nunique():,} write-in invoices")
print()

print("--- Group 2: has Customer ID (free item candidates) ---")
group2 = zero_price_pos_qty.loc[has_customer]
print(f"Rows: {len(group2):,}")
print("Description value counts:")
print(group2["Description"].value_counts(dropna=False).head(15))
print()
print(group2[["Invoice", "StockCode", "Description", "Quantity", "Customer ID"]].head(15))

# Group 1 mirrors is_stock_adjustment (which covers negative-quantity write-offs): no customer,
# no price, generic/blank operational notes -- stock being corrected back into inventory rather
# than sold. Group 2 has a real Customer ID attached to a real product -- a genuine transaction,
# just given away for free -- so it's kept in sales/orders rather than excluded.
df["is_stock_writein"] = zero_price_pos_qty_mask & df["Customer ID"].isna()
df["is_free_item"] = zero_price_pos_qty_mask & df["Customer ID"].notna()

print()
print("Flagged as is_stock_writein:", int(df["is_stock_writein"].sum()))
print("Flagged as is_free_item:", int(df["is_free_item"].sum()))

Zero-price, positive-quantity, non-cancelled, non-stock-adjustment rows: 2,628

Split by Customer ID presence:
Customer ID
False    2560
True       68
Name: count, dtype: int64

--- Group 1: no Customer ID (stock write-in candidates) ---
Rows: 2,560
Description value counts:
Description
NaN                                    1641
check                                    39
found                                    28
OWL DOORSTOP                             14
adjustment                               14
POLYESTER FILLER PAD 45x45cm             11
POLYESTER FILLER PAD 40x40cm             10
?                                         9
Found                                     9
FRENCH BLUE METAL DOOR SIGN 1             9
PICNIC BASKET WICKER LARGE                8
AIRLINE BAG VINTAGE WORLD CHAMPION        8
BOX OF 24 COCKTAIL PARASOLS               8
MINT KITCHEN SCALES                       8
RECIPE BOX PANTRY YELLOW DESIGN           8
Name: count, dtype: int64



Write-in rows sharing an invoice with a normally priced row: 70 of 1,924 write-in invoices



--- Group 2: has Customer ID (free item candidates) ---
Rows: 68
Description value counts:
Description
Manual                               7
CHRISTMAS PUDDING TRINKET POT        2
REGENCY CAKESTAND 3 TIER             2
6 RIBBONS EMPIRE                     1
DOOR MAT FAIRY CAKE                  1
CHRISTMAS CRAFT WHITE FAIRY          1
ANTIQUE LILY FAIRY LIGHTS            1
ANTIQUE GLASS HEART DECORATION       1
 FLAMINGO LIGHTS                     1
CHARLOTTE BAG , SUKI DESIGN          1
RETRO SPOT LARGE MILK JUG            1
VINTAGE GLASS COFFEE CADDY           1
CAST IRON HOOK GARDEN TROWEL         1
CAST IRON HOOK GARDEN FORK           1
AIRLINE BAG VINTAGE JET SET WHITE    1
Name: count, dtype: int64

       Invoice StockCode                        Description  Quantity  \
4674    489825     22076                 6 RIBBONS EMPIRE          12   
6781    489998     48185                DOOR MAT FAIRY CAKE         2   
16107   490727         M                             Manual     

## 8d. Analyze "Adjust Bad Debt" Rows (Invoice Prefix "A")

In [16]:
# Neither is_cancelled ("C" prefix) nor is_stock_adjustment (negative quantity) catches these --
# invoices prefixed "A" are financial bad-debt write-offs (StockCode "B"), not a stock movement
# or a customer cancellation, and were previously sitting silently inside sales_rows/gross_revenue.
df["is_bad_debt_adjustment"] = df["Invoice"].str.upper().str.startswith("A", na=False)
bad_debt_rows = df.loc[df["is_bad_debt_adjustment"]].copy()

print(f"Bad debt adjustment rows: {len(bad_debt_rows)}")
print(bad_debt_rows[["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "InvoiceDate"]].to_string(index=False))
print()
print("Net revenue impact:", round(bad_debt_rows["Revenue"].sum(), 2))
print()
print("Months containing a bad debt write-off:", sorted(bad_debt_rows["InvoiceDate"].dt.to_period("M").astype(str).unique()))

Bad debt adjustment rows: 6
Invoice StockCode     Description  Quantity     Price  Customer ID         InvoiceDate
A506401         B Adjust bad debt         1 -53594.36          NaN 2010-04-29 13:36:00
A516228         B Adjust bad debt         1 -44031.79          NaN 2010-07-19 11:24:00
A528059         B Adjust bad debt         1 -38925.87          NaN 2010-10-20 12:04:00
A563185         B Adjust bad debt         1  11062.06          NaN 2011-08-12 14:50:00
A563186         B Adjust bad debt         1 -11062.06          NaN 2011-08-12 14:51:00
A563187         B Adjust bad debt         1 -11062.06          NaN 2011-08-12 14:52:00

Net revenue impact: -147614.08

Months containing a bad debt write-off: ['2010-04', '2010-07', '2010-10', '2011-08']


## 8e. Full Data Taxonomy (Cross-Examination of All Record Categories)

Each row is assigned to exactly one category by checking the rules below **in order** — a row gets the first category whose rule it matches (implemented as the if/elif chain in `categorize()` below). Anything matching none of the first five rules falls through to `normal_sale`.

| Priority | Category | Rule | Counted as a sale? |
|---|---|---|---|
| 1 | `cancelled` | `Invoice` starts with "C" | No — excluded from `sales_rows`, but netted back in for net revenue (§10) |
| 2 | `bad_debt_adjustment` | `Invoice` starts with "A" | No — excluded entirely |
| 3 | `stock_adjustment` | `Quantity < 0` (and not already `cancelled`/`bad_debt_adjustment`) | No — excluded entirely |
| 4 | `stock_writein` | `Price = 0` and `Quantity > 0` and `Customer ID` missing | No — excluded entirely |
| 5 | `free_item` | `Price = 0` and `Quantity > 0` and `Customer ID` present | **Yes** — kept in, real order at £0 |
| 6 *(fallback)* | `normal_sale` | Everything left over (in practice, `Quantity > 0` and `Price > 0`) | **Yes** |

In [17]:
def sign(series):
    return pd.cut(
        series,
        bins=[-float("inf"), -1e-9, 1e-9, float("inf")],
        labels=["negative", "zero", "positive"],
    )


def categorize(row):
    if row["is_cancelled"]:
        return "cancelled"
    if row["is_bad_debt_adjustment"]:
        return "bad_debt_adjustment"
    if row["is_stock_adjustment"]:
        return "stock_adjustment"
    if row["is_stock_writein"]:
        return "stock_writein"
    if row["is_free_item"]:
        return "free_item"
    return "normal_sale"


df["qty_sign"] = sign(df["Quantity"])
df["price_sign"] = sign(df["Price"])
df["category"] = df.apply(categorize, axis=1)

data_taxonomy = (
    df.groupby(["category", "qty_sign", "price_sign"], observed=True)
    .size()
    .reset_index(name="count")
    .sort_values(["category", "count"], ascending=[True, False])
)

print(f"Total rows across all categories: {data_taxonomy['count'].sum():,} (dataset has {len(df):,})")
data_taxonomy

Total rows across all categories: 1,044,831 (dataset has 1,044,831)


,category,qty_sign,price_sign,count
0,bad_debt_adjustment,positive,negative,5
1,bad_debt_adjustment,positive,positive,1
2,cancelled,negative,positive,19160
3,cancelled,positive,positive,1
4,free_item,positive,zero,68
5,normal_sale,positive,positive,1019643
6,stock_adjustment,negative,zero,3393
7,stock_writein,positive,zero,2560


## 8f. Duplicate Line-Item Flag (`is_duplicate_line`)

In [18]:
# Full-row duplicates: same Invoice, StockCode, Description, Quantity, InvoiceDate, Price,
# Customer ID, and Country. The sheet-overlap duplication (§3b) is already removed, so what's
# left here is duplication WITHIN a single sheet -- e.g. the same product logged as two separate
# single-unit lines rather than one line at a higher quantity. This flag doesn't drop anything;
# it's for visibility, since it isn't clear this is an error rather than how orders were entered.
dedup_cols = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID", "Country"]
df["is_duplicate_line"] = df.duplicated(subset=dedup_cols, keep=False)

duplicate_lines = df.loc[df["is_duplicate_line"]].copy()
print(f"Duplicate-flagged rows: {len(duplicate_lines):,}")
print(f"Distinct duplicate-content groups: {duplicate_lines.drop_duplicates(subset=dedup_cols).shape[0]:,}")
print(f"Distinct invoices affected: {duplicate_lines['Invoice'].nunique():,}")
print()

group_sizes = duplicate_lines.groupby(dedup_cols, observed=True).size()
print("How many times each duplicate group repeats:")
print(group_sizes.value_counts().sort_index())
print()

extra_copies = duplicate_lines.copy()
extra_copies["copy_number"] = extra_copies.groupby(dedup_cols, observed=True).cumcount()
extra_only = extra_copies.loc[extra_copies["copy_number"] > 0]
print(f"Revenue represented by the 'extra' (2nd+) copies in each group: £{extra_only['Revenue'].sum():,.2f}")
print(f"Duplicate rows that are also cancellations: {duplicate_lines['is_cancelled'].sum():,} of {len(duplicate_lines):,}")
print()

print("Quantity distribution among duplicated lines:")
print(duplicate_lines["Quantity"].describe())
print(f"Duplicate lines with Quantity == 1: {(duplicate_lines['Quantity'] == 1).sum():,} of {len(duplicate_lines):,}")
print()
print("Customers with the most duplicate-flagged rows (likely wholesale/bulk buyers logging repeat single-unit lines):")
print(duplicate_lines["Customer ID"].value_counts(dropna=False).head(10))

# Is "wholesale/bulk buyer" a fair read, or just noise from a handful of customers with lots of
# orders generally? Rank every customer by total revenue and total order count, then check where
# the top duplicate-line customers fall in that ranking -- if they're wholesale buyers, they
# should sit at the extreme high end on both measures, not just have a lot of duplicate rows.
customer_profile = df.groupby("Customer ID").agg(
    total_rows=("Revenue", "size"),
    total_revenue=("Revenue", "sum"),
    total_orders=("Invoice", "nunique"),
).reset_index()
customer_profile["revenue_percentile"] = customer_profile["total_revenue"].rank(pct=True)
customer_profile["orders_percentile"] = customer_profile["total_orders"].rank(pct=True)

top_duplicate_customers = duplicate_lines["Customer ID"].value_counts().head(10).index.dropna()
wholesale_check = (
    customer_profile.loc[customer_profile["Customer ID"].isin(top_duplicate_customers)]
    .merge(duplicate_lines["Customer ID"].value_counts().rename("duplicate_rows"), left_on="Customer ID", right_index=True)
    .sort_values("total_revenue", ascending=False)
)
print()
print("Where the top duplicate-line customers rank among ALL customers by revenue and order count:")
print(wholesale_check.to_string(index=False))

# Are duplicates confined to a narrow window (pointing at a system bug active only then), or
# spread across the whole dataset (more consistent with an ongoing order-entry practice)?
df["month"] = df["InvoiceDate"].dt.to_period("M").astype("string")
duplicate_lines["month"] = duplicate_lines["InvoiceDate"].dt.to_period("M").astype("string")

monthly_duplicates = pd.concat(
    [df.groupby("month").size().rename("total_rows"), duplicate_lines.groupby("month").size().rename("duplicate_rows")],
    axis=1,
).fillna(0)
monthly_duplicates["duplicate_rate_pct"] = (monthly_duplicates["duplicate_rows"] / monthly_duplicates["total_rows"] * 100).round(2)

print()
print("Duplicate rows by month:")
print(monthly_duplicates.to_string())
print()
print("Duplicate row date range:", duplicate_lines["InvoiceDate"].min(), "->", duplicate_lines["InvoiceDate"].max())
print("Full dataset date range: ", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())
print("Months with zero duplicate rows:", (monthly_duplicates["duplicate_rows"] == 0).sum(), "of", len(monthly_duplicates))
# Same question at day-level granularity: is there one specific day dominating (a one-off
# system incident) or is it spread across nearly every day the business operated?
date_counts = duplicate_lines["InvoiceDate"].dt.date.value_counts()
print()
print(f"Dates with at least one duplicate row: {len(date_counts)} of {df['InvoiceDate'].dt.date.nunique()} total dates in the dataset")
print()
print("Top 20 dates by duplicate row count:")
print(date_counts.head(20))
print()
print("Distribution of duplicate-row-count per date:")
print(date_counts.describe())

Duplicate-flagged rows: 22,813
Distinct duplicate-content groups: 11,001
Distinct invoices affected: 4,387

How many times each duplicate group repeats:
2     10231
3       540
4        78
5        11
6        10
8         1
12        1
20        1
Name: count, dtype: int64

Revenue represented by the 'extra' (2nd+) copies in each group: £53,677.75
Duplicate rows that are also cancellations: 116 of 22,813

Quantity distribution among duplicated lines:
count    22813.000000
mean         2.728663
std         19.749204
min      -1296.000000
25%          1.000000
50%          1.000000
75%          2.000000
max       1440.000000
Name: Quantity, dtype: float64
Duplicate lines with Quantity == 1: 16,015 of 22,813

Customers with the most duplicate-flagged rows (likely wholesale/bulk buyers logging repeat single-unit lines):
Customer ID
12748.0    527
17841.0    515
16549.0    300
16782.0    291
NaN        264
14606.0    169
16686.0    153
13230.0    141
17920.0    129
18022.0    127
Name: cou


Where the top duplicate-line customers rank among ALL customers by revenue and order count:
 Customer ID  total_rows  total_revenue  total_orders  revenue_percentile  orders_percentile  duplicate_rows
     17841.0       12907       67984.13           289            0.996801           0.999663             515
     12748.0        6936       47976.73           365            0.994613           0.999832             527
     14606.0        6587       29500.40           259            0.991077           0.999327             169
     17920.0        1711       23774.00            56            0.988215           0.989815             129
     16549.0        3255       13159.64            37            0.973569           0.975084             300
     16984.0        1160       10599.42            19            0.962458           0.919781             118
     16782.0        1849        9983.21            50            0.959428           0.986111             291
     13230.0        1327        622


Duplicate rows by month:
         total_rows  duplicate_rows  duplicate_rate_pct
month                                                  
2009-12       45222             977                2.16
2010-01       31551             628                1.99
2010-02       29386             631                2.15
2010-03       41511            1014                2.44
2010-04       34056             801                2.35
2010-05       35323             814                2.30
2010-06       39983            1004                2.51
2010-07       33383             766                2.29
2010-08       33306             708                2.13
2010-09       42091             940                2.23
2010-10       59094            1636                2.77
2010-11       78015            2747                3.52
2010-12       42481             952                2.24
2011-01       35147             475                1.35
2011-02       27707             445                1.61
2011-03       36748   

count    595.000000
mean      38.341176
std       36.313815
min        2.000000
25%       14.000000
50%       27.000000
75%       50.000000
max      257.000000
Name: count, dtype: float64


## 8g. Non-Product Stock Code Flag (`is_non_product_code`)

In [19]:
# Standard product codes are 5 digits with an optional short letter suffix (e.g. "85123A").
# Anything else is a bookkeeping/operational code riding along in the same StockCode column --
# postage, manual adjustments, discounts, fees, gift vouchers, test rows, etc.
df["is_non_product_code"] = ~df["StockCode"].str.match(r"^\d{5}[A-Za-z]{0,4}$", na=False)

# category (from Section 8e) already encodes which flag -- if any -- takes a row out of
# sales_rows: "normal_sale" and "free_item" are counted as a sale; every other category is
# excluded.
non_product = df.loc[df["is_non_product_code"]].copy()
non_product["counted_as_sale"] = non_product["category"].isin(["normal_sale", "free_item"])

print(f"Non-product-code rows: {len(non_product):,} across {non_product['StockCode'].nunique()} distinct codes")
print()

code_breakdown = (
    non_product.groupby(["StockCode", "category"], observed=True)
    .agg(rows=("StockCode", "size"), revenue=("Revenue", "sum"))
    .reset_index()
    .sort_values(["StockCode", "rows"], ascending=[True, False])
)
print(code_breakdown.to_string(index=False))
print()

print("Rolled up by StockCode -- total rows, net revenue, and how much of that revenue currently")
print("counts as a sale vs. is excluded by an existing flag:")
rollup = non_product.groupby("StockCode", observed=True).apply(
    lambda g: pd.Series(
        {
            "rows": len(g),
            "net_revenue": g["Revenue"].sum(),
            "revenue_counted_as_sale": g.loc[g["counted_as_sale"], "Revenue"].sum(),
            "revenue_excluded": g.loc[~g["counted_as_sale"], "Revenue"].sum(),
        }
    ),
    include_groups=False,
).sort_values("rows", ascending=False)
print(rollup.to_string())

Non-product-code rows: 5,976 across 61 distinct codes

   StockCode            category  rows     revenue
     47503J          normal_sale     1      16.130
      ADJUST         normal_sale    36    8897.930
      ADJUST           cancelled    31   -2062.690
     ADJUST2         normal_sale     3     731.050
   AMAZONFEE           cancelled    33 -241988.300
   AMAZONFEE         normal_sale     3   20467.800
           B bad_debt_adjustment     6 -147614.080
BANK CHARGES           cancelled    66  -36001.490
BANK CHARGES         normal_sale    34     519.241
          C2         normal_sale   267   13426.000
          C2           cancelled     7    -290.000
          C2       stock_writein     3       0.000
          C3    stock_adjustment     1       0.000
        CRUK           cancelled    16   -7933.430
           D           cancelled   168  -13277.520
           D         normal_sale     5     397.890
    DCGS0003         normal_sale    13      32.590
    DCGS0003    stock_adjus

In [20]:
# Group the near-duplicate codes together (M/m are the same thing typed differently, likewise
# ADJUST/ADJUST2). TEST001/TEST002 no longer appear here -- they were dropped from df entirely
# in §6b as literal QA rows, not real transactions.
code_groups = {
    "M": "M / m", "m": "M / m",
    "ADJUST": "ADJUST / ADJUST2", "ADJUST2": "ADJUST / ADJUST2",
}
non_product["code_group"] = non_product["StockCode"].map(lambda c: code_groups.get(c, c))

named_groups = [
    "POST", "DOT", "M / m", "C2", "D", "BANK CHARGES", "S",
    "ADJUST / ADJUST2", "AMAZONFEE", "CRUK", "B",
]
named = non_product.loc[non_product["code_group"].isin(named_groups)]
rest = non_product.loc[~non_product["code_group"].isin(named_groups)]

category_pivot = (
    named.groupby(["code_group", "category"], observed=True)
    .agg(rows=("Revenue", "size"), revenue=("Revenue", "sum"))
    .round(2)
)
print("Rows / revenue per named code, split by category (matches the §7.8 table in the report):")
for group in named_groups:
    print(f"--- {group} ---")
    print(category_pivot.loc[group].to_string())
    print()

print(f"--- remaining {rest['StockCode'].nunique()} codes, combined ---")
print(rest.groupby("category", observed=True).agg(rows=("Revenue", "size"), revenue=("Revenue", "sum")).round(2).to_string())

# The most common raw Description text per named code group -- backs the "Raw Description text"
# column of the §7.8 table (e.g. POST -> "POSTAGE", M / m -> "Manual").
print()
print("Most common raw Description text per named code group:")
print(named.groupby("code_group", observed=True)["Description"].agg(lambda s: s.value_counts().index[0]).to_string())

Rows / revenue per named code, split by category (matches the §7.8 table in the report):
--- POST ---
               rows    revenue
category                      
cancelled       228  -15252.01
normal_sale    1851  125682.42
stock_writein     7       0.00

--- DOT ---
               rows    revenue
category                      
cancelled         3     -10.01
normal_sale    1415  309854.11
stock_writein     7       0.00

--- M / m ---
             rows    revenue
category                    
cancelled     535 -422565.51
free_item       7       0.00
normal_sale   861  339629.94

--- C2 ---
               rows  revenue
category                    
cancelled         7   -290.0
normal_sale     267  13426.0
stock_writein     3      0.0

--- D ---
             rows   revenue
category                   
cancelled     168 -13277.52
normal_sale     5    397.89

--- BANK CHARGES ---
             rows   revenue
category                   
cancelled      66 -36001.49
normal_sale    34    519.24



## 8h. Multiple Descriptions per Product Code (`has_variant_description`)

In [21]:
# Restricted to genuine product codes (is_non_product_code == False) -- non-product codes like
# "ADJUST"/"M" are *expected* to carry many different free-text notes by design (§7.8), so
# including them here would just be noise, not a data-quality finding about products.
product_rows = df.loc[~df["is_non_product_code"]].dropna(subset=["Description"])

n_distinct = product_rows.groupby("StockCode")["Description"].nunique()
multi_desc_codes = n_distinct.loc[n_distinct > 1].index

print(f"Product codes with more than one distinct Description: {len(multi_desc_codes):,} of {n_distinct.shape[0]:,}")

def modal_share(descriptions):
    counts = descriptions.value_counts()
    return counts.iloc[0] / counts.sum()

shares = (
    product_rows.loc[product_rows["StockCode"].isin(multi_desc_codes)]
    .groupby("StockCode")["Description"]
    .apply(modal_share)
)

# A code where one wording covers >=95% of its rows is one real product name plus a handful of
# stray operational notes ("missing", "found", "wrongly coded-23343", ...) riding in the
# Description field -- not a naming problem. Below that threshold the wordings are genuinely
# competing, which usually means the product was renamed/reworded partway through the two-year
# window (e.g. "PINK POLKADOT PLATE" vs "PINK SPOTTY PLATE" for the same StockCode).
stray_notes = shares.loc[shares >= 0.95]
renamed_products = shares.loc[shares < 0.95].sort_values()

df["has_variant_description"] = df["StockCode"].isin(renamed_products.index)

print(f"  - one dominant name + stray one-off notes: {len(stray_notes):,} codes")
print(f"  - meaningfully split between multiple wordings (likely renamed/reworded product): {len(renamed_products):,} codes")
print()

stray_note_sizes = (
    product_rows.loc[product_rows["StockCode"].isin(stray_notes.index)]
    .groupby("StockCode").size().sort_values(ascending=False)
)
stray_examples = (
    product_rows.loc[product_rows["StockCode"].isin(stray_note_sizes.index[:15])]
    .groupby("StockCode")["Description"]
    .apply(lambda s: s.value_counts().to_dict())
    .loc[stray_note_sizes.index[:15]]
    .rename("descriptions")
    .reset_index()
)
print("Largest stray-note codes (one dominant name + a few one-off notes) -- top 15 by row count:")
display(stray_examples.head(15))

description_variants = (
    product_rows.loc[product_rows["StockCode"].isin(renamed_products.index)]
    .groupby("StockCode")["Description"]
    .apply(lambda s: s.value_counts().to_dict())
    .loc[renamed_products.index]
    .rename("descriptions")
    .reset_index()
)
print("Most evenly split codes (same product, different wording over time) -- top 15:")
display(description_variants.head(15))

# Resolve each ambiguous code to a single canonical Description, using a different rule per
# group since they represent different situations:
#   - stray-note codes: the dominant wording IS the product name, so use the most frequent
#     (modal) Description and discard the one-off notes.
#   - renamed/reworded codes: there's no single "correct" wording, so use whichever Description
#     was actually in use most recently (latest InvoiceDate), since that reflects current
#     catalog naming rather than an arbitrary pick.
modal_description = (
    product_rows.loc[product_rows["StockCode"].isin(stray_notes.index)]
    .groupby("StockCode")["Description"]
    .agg(lambda s: s.value_counts().idxmax())
)
latest_description = (
    product_rows.loc[product_rows["StockCode"].isin(renamed_products.index)]
    .sort_values("InvoiceDate")
    .groupby("StockCode")["Description"]
    .last()
)
canonical_description = pd.concat([modal_description, latest_description])

original_description = df["Description"]
mapped_description = df["StockCode"].map(canonical_description)
rows_relabelled = (mapped_description.notna() & (mapped_description != original_description)).sum()
df["Description"] = mapped_description.fillna(original_description)

print(
    f"Rows relabelled with a canonical Description: {rows_relabelled:,} across "
    f"{len(canonical_description):,} StockCodes ({len(modal_description):,} resolved to the "
    f"most-frequent wording, {len(latest_description):,} resolved to the latest wording)"
)

Product codes with more than one distinct Description: 1,224 of 4,907


  - one dominant name + stray one-off notes: 687 codes
  - meaningfully split between multiple wordings (likely renamed/reworded product): 537 codes

Largest stray-note codes (one dominant name + a few one-off notes) -- top 15 by row count:


,StockCode,level_1,descriptions
0,85123A,GIN + TONIC DIET METAL SIGN,NaN
1,85123A,GIN AND TONIC DIET METAL SIGN,NaN
2,85123A,STRAWBERRY CERAMIC TRINKET BOX,NaN
3,85123A,STRAWBERRY CERAMIC TRINKET POT,NaN
4,85123A,RED HANGING HEART T-LIGHT HOLDER,NaN
5,85123A,85123a mixed,NaN
6,85123A,JUMBO BAG PINK VINTAGE PAISLEY,NaN
7,85123A,?,1.0
8,85123A,6 RIBBONS RUSTIC CHARM,NaN
9,85123A,found,NaN


Most evenly split codes (same product, different wording over time) -- top 15:


,StockCode,level_1,descriptions
0,84997A,BLUE POLKADOT GARDEN PARASOL,NaN
1,84997A,BLUE WHITE SPOTS GARDEN PARASOL,NaN
2,84997A,wet/rusty,NaN
3,84997A,PINK POLKADOT GARDEN PARASOL,NaN
4,84997A,PINK WHITE SPOTS GARDEN PARASOL,NaN
5,84997A,ANIMAL STICKERS,NaN
6,84997A,ANIMAL STICKERS,NaN
7,84997A,FOOD/DRINK SPONGE STICKERS,NaN
8,84997A,FOOD/DRINK SPUNGE STICKERS,NaN
9,84997A,FLOWERS HANDBAG blue and orange,NaN


Rows relabelled with a canonical Description: 77,528 across 1,224 StockCodes (687 resolved to the most-frequent wording, 537 resolved to the latest wording)


## 8i. Non-Country Flag (`is_non_country`)

In [22]:
# A handful of "Country" values aren't actual countries: "Unspecified" is a genuine unknown,
# while "European Community", "Channel Islands", and "West Indies" are a political bloc, a
# dependency, and a multi-country region respectively -- none map cleanly onto a single nation.
# ("EIRE" and "RSA" ARE real countries -- just Irish/Afrikaans-derived names for Ireland and
# South Africa -- so they're left alone.)
non_country_values = ["Unspecified", "European Community", "Channel Islands", "West Indies"]
df["is_non_country"] = df["Country"].isin(non_country_values)

print(f"Non-country rows: {df['is_non_country'].sum():,} ({df['is_non_country'].mean() * 100:.2f}% of rows)")
display(
    df.loc[df["is_non_country"]]
    .groupby("Country", observed=True)
    .agg(rows=("Country", "size"), revenue=("Revenue", "sum"))
    .sort_values("rows", ascending=False)
)

# Flagged for visibility only -- the revenue is real, so these rows stay in Top Countries (§6)
# under their own label rather than being dropped or forced into a single-country bucket.

Non-country rows: 2,518 (0.24% of rows)


,rows,revenue
Country,,
Channel Islands,1647,41090.08
Unspecified,756,9687.32
European Community,61,1291.75
West Indies,54,536.41


## 9. Review the First Rows and Missing Values

In [23]:
display(df.head(10))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))
display(df["Country"].value_counts(dropna=False).head(20).to_frame("row_count"))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,Revenue,is_cancelled,is_stock_adjustment,is_stock_writein,is_free_item,is_bad_debt_adjustment,qty_sign,price_sign,category,is_duplicate_line,month,is_non_product_code,has_variant_description,is_non_country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010,83.4,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False,False,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,81.0,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False,False,False
2,489434,79323W,"Unsaleable, destroyed.",12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,81.0,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False,True,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010,100.8,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010,30.0,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False,False,False
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,Year 2009-2010,39.6,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False,False,False
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010,30.0,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False,False,False
7,489434,21523,DOORMAT FANCY FONT HOME SWEET HOME,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,Year 2009-2010,59.5,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False,True,False
8,489435,22350,ILLUSTRATED CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,Year 2009-2010,30.6,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False,True,False
9,489435,22349,DOG BOWL CHASING BALL DESIGN,12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom,Year 2009-2010,45.0,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False,True,False


,missing_values
Customer ID,235286
Description,3340
Invoice,0
StockCode,0
Quantity,0
InvoiceDate,0
Price,0
Country,0
source_sheet,0
Revenue,0


,row_count
Country,
United Kingdom,959966
EIRE,17689
Germany,17363
France,14059
Netherlands,5138
Spain,3766
Switzerland,3183
Belgium,3111
Portugal,2540


## 10. Aggregate Sales Metrics

In [24]:
non_transactional = df["is_stock_adjustment"] | df["is_stock_writein"] | df["is_bad_debt_adjustment"]

# is_free_item is NOT excluded here -- it's a real order for a real customer, just at £0.
sales_rows = df.loc[~df["is_cancelled"] & ~non_transactional & df["Revenue"].notna()].copy()
sales_rows["month"] = sales_rows["InvoiceDate"].dt.to_period("M").astype("string")

# Cancellations are not reliably linkable to a specific original order (see §7.1 decision),
# but the money they return is real, so revenue must be NET of cancellations rather than
# computed on sales rows alone -- otherwise cancelled purchases are counted as revenue that
# was never actually kept. revenue_rows = sales + cancellations, non-transactional rows
# (stock adjustments/write-ins/bad debt) excluded since they aren't customer transactions.
revenue_rows = df.loc[~non_transactional & df["Revenue"].notna()].copy()
revenue_rows["month"] = revenue_rows["InvoiceDate"].dt.to_period("M").astype("string")

gross_revenue = sales_rows["Revenue"].sum()
cancellation_revenue = df.loc[df["is_cancelled"], "Revenue"].sum()
bad_debt_revenue = df.loc[df["is_bad_debt_adjustment"], "Revenue"].sum()
net_revenue = revenue_rows["Revenue"].sum()

monthly_revenue = revenue_rows.groupby("month", as_index=False)["Revenue"].sum().sort_values("month")
top_products = (
    revenue_rows.groupby("Description", dropna=False)["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .rename("Revenue")
    .reset_index()
)
top_countries = (
    revenue_rows.groupby("Country", dropna=False)["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .rename("Revenue")
    .reset_index()
)
top_countries["share_of_net_revenue_pct"] = (top_countries["Revenue"] / net_revenue * 100).round(1)

metrics = pd.Series(
    {
        "gross_revenue": gross_revenue,
        "cancellation_revenue": cancellation_revenue,
        "bad_debt_revenue": bad_debt_revenue,
        "net_revenue": net_revenue,
        "orders": sales_rows["Invoice"].nunique(),
        "customers": revenue_rows["Customer ID"].nunique(),
        "cancelled_rows": int(df["is_cancelled"].sum()),
        "stock_adjustment_rows": int(df["is_stock_adjustment"].sum()),
        "stock_writein_rows": int(df["is_stock_writein"].sum()),
        "free_item_rows": int(df["is_free_item"].sum()),
        "bad_debt_rows": int(df["is_bad_debt_adjustment"].sum()),
        "duplicate_flagged_rows": int(df["is_duplicate_line"].sum()),
        "non_product_code_rows": int(df["is_non_product_code"].sum()),
        "flagged_rows": len(invalid_rows),
    },
    name="value",
)
display(metrics.to_frame())
print("Net revenue by month (full 25 months -- backs the §4 seasonality claims):")
display(monthly_revenue)
display(top_products)

# How much of the top-20 products total is shipping (POST/DOT), not merchandise?
postage_revenue = top_products.loc[top_products["Description"].isin(["POSTAGE", "DOTCOM POSTAGE"]), "Revenue"].sum()
print(f"POSTAGE + DOTCOM POSTAGE combined: £{postage_revenue:,.0f} ({postage_revenue / top_products['Revenue'].sum() * 100:.1f}% of the top-20 total)")

display(top_countries)

,value
gross_revenue,2.052245e+07
cancellation_revenue,-1.465281e+06
bad_debt_revenue,-1.476141e+05
net_revenue,1.905717e+07
orders,4.007000e+04
customers,5.940000e+03
cancelled_rows,1.916100e+04
stock_adjustment_rows,3.393000e+03
stock_writein_rows,2.560000e+03
free_item_rows,6.800000e+01


Net revenue by month (full 25 months -- backs the §4 seasonality claims):


,month,Revenue
0,2009-12,799733.610
1,2010-01,623942.892
2,2010-02,533091.426
3,2010-03,765848.761
4,2010-04,644152.292
5,2010-05,615322.830
6,2010-06,679786.610
7,2010-07,619268.150
8,2010-08,656776.340
9,2010-09,853650.431


,Description,Revenue
0,REGENCY CAKESTAND 3 TIER,314513.47
1,DOTCOM POSTAGE,309844.10
2,WHITE HANGING HEART T-LIGHT HOLDER,252122.83
3,JUMBO BAG RED RETROSPOT,180830.85
4,PARTY BUNTING,147157.43
5,ASSORTED COLOUR BIRD ORNAMENT,128907.01
6,PAPER CHAIN KIT 50'S CHRISTMAS,116422.49
7,POSTAGE,110430.41
8,CHILLI LIGHTS,80237.48
9,POPCORN HOLDER,78935.92


POSTAGE + DOTCOM POSTAGE combined: £420,275 (17.6% of the top-20 total)


,Country,Revenue,share_of_net_revenue_pct
0,United Kingdom,1.618601e+07,84.9
1,EIRE,6.102443e+05,3.2
2,Netherlands,5.483323e+05,2.9
3,Germany,4.125178e+05,2.2
4,France,3.219285e+05,1.7
5,Australia,1.665119e+05,0.9
6,Switzerland,9.942536e+04,0.5
7,Spain,9.106476e+04,0.5
8,Sweden,8.780942e+04,0.5
9,Denmark,6.445959e+04,0.3


## 11. Save Processed Outputs


In [25]:
# Deliberately empty: this notebook does not write any processed/cleaned dataset to disk.
# It is exploration-only, per the pipeline design (see the intro cell) -- its job is to reach
# and document decisions, not to produce a file the production pipeline depends on. Every
# decision made above is instead re-implemented as a Power Query step in
# power_query_cleaning_guide.md, which the live Power BI model actually runs against the raw
# monthly CSVs on every refresh.
